<a href="https://colab.research.google.com/github/jannikm81-gif/Humanitas-purissima-/blob/main/notebooks/exile_valve_verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import sys
# Falls PyNaCl (nacl) fehlt, installieren wir es im Notebook
try:
    import nacl.signing
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pynacl"])
    import nacl.signing

import json
import time

def generate_offline_voucher(issuer_private_key, dissident_id, resource_units, expiry_hours=72):
    """
    Erstellt einen kryptografisch signierten Offline-Voucher (Fluchthilfe-Ticket).
    """
    timestamp = int(time.time())
    expiry_time = timestamp + (expiry_hours * 3600)

    # Der payload enthält die unumstößlichen Fakten des Vouchers
    payload = {
        "protocol": "HP-v10-Ebene0",
        "dissident_id": dissident_id,
        "resource_units": resource_units,
        "expires_at": expiry_time
    }

    # In String konvertieren für die Signierung
    serialized_payload = json.dumps(payload, sort_keys=True).encode('utf-8')

    # Digitale Signatur mit dem privaten Schlüssel erstellen
    signature = issuer_private_key.sign(serialized_payload)

    # Das Ticket
    voucher_ticket = {
        "payload": payload,
        "signature_hex": signature.signature.hex()
    }

    return voucher_ticket

def verify_offline_voucher(voucher_ticket, issuer_public_key):
    """
    Ein isoliertes Grenz-Sub-Grid prüft den Voucher rein mathematisch.
    """
    payload = voucher_ticket["payload"]
    signature = bytes.fromhex(voucher_ticket["signature_hex"])
    serialized_payload = json.dumps(payload, sort_keys=True).encode('utf-8')

    current_time = int(time.time())

    # 1. Prüfung: Ist der Voucher zeitlich noch gültig?
    if current_time > payload["expires_at"]:
        return False, "Voucher abgelaufen (Exil-Zeitfenster überschritten)"

    # 2. Prüfung: Stimmt die mathematische Signatur des ausstellenden Knotens?
    try:
        issuer_public_key.verify(serialized_payload, signature)
        return True, f"Voucher VALIDIERT. Gewähre {payload['resource_units']} Einheiten Grundversorgung."
    except nacl.exceptions.BadSignatureError:
        return False, "Kryptografische Fälschung erkannt! Signatur ungültig."

# --- SIMULATION DES EXIL-VENTILS ---

# 1. Schlüsselpaar generieren
sk_subgrid = nacl.signing.SigningKey.generate()
pk_subgrid = sk_subgrid.verify_key

# 2. Ein Dissident flieht und erhält offline einen Voucher für die Fluchthilfe
print("=== HP-v10: EXIL-VENTIL OFFLINE-GENERIERUNG ===")
dissident_wallet = "HP-SSI-USER-JANNIK-M"
voucher = generate_offline_voucher(sk_subgrid, dissident_wallet, resource_units=150)
print(json.dumps(voucher, indent=2))
print("-" * 60)

# 3. Das Grenz-Sub-Grid verifiziert offline
print("=== OFFLINE-VERIFIKATION IM EXIL-ZIEL ===")
success, message = verify_offline_voucher(voucher, pk_subgrid)
print(f"Status: {success}")
print(f"System-Meldung: {message}")
print("-" * 60)

# 4. Simulation eines Angriffs (Fälschungsversuch)
print("=== ADVERSARIELLER ANGRIFFSTEST (FÄLSCHUNGS-VERSUCH) ===")
forged_voucher = json.loads(json.dumps(voucher))
forged_voucher["payload"]["resource_units"] = 99999

attack_success, attack_message = verify_offline_voucher(forged_voucher, pk_subgrid)
print(f"Status: {attack_success}")
print(f"System-Meldung: {attack_message}")

=== HP-v10: EXIL-VENTIL OFFLINE-GENERIERUNG ===
{
  "payload": {
    "protocol": "HP-v10-Ebene0",
    "dissident_id": "HP-SSI-USER-JANNIK-M",
    "resource_units": 150,
    "expires_at": 1791835214
  },
  "signature_hex": "b2e45925237902306d57b1fda10228eb99dbbedd41b6909ecfce40558102f2dda8a4d549cefc946ee87f8d882f170cda1ebc782fcbb99a1c4f86178f086d890f"
}
------------------------------------------------------------
=== OFFLINE-VERIFIKATION IM EXIL-ZIEL ===
Status: True
System-Meldung: Voucher VALIDIERT. Gewähre 150 Einheiten Grundversorgung.
------------------------------------------------------------
=== ADVERSARIELLER ANGRIFFSTEST (FÄLSCHUNGS-VERSUCH) ===
Status: False
System-Meldung: Kryptografische Fälschung erkannt! Signatur ungültig.
